# 📊 Task 5: Data Visualization Portfolio — COVID-19 Dataset
**Student Project | Data Science Fundamentals**

| # | Chart Type | Library |
|---|---|---|
| 1 | Line Plot (Time Series) | Matplotlib |
| 2 | Multi-Line Plot | Matplotlib |
| 3 | Bar Chart | Matplotlib |
| 4 | Horizontal Bar Chart | Matplotlib |
| 5 | Histogram | Matplotlib |
| 6 | Scatter Plot (Correlation) | Matplotlib |
| 7 | Box Plot | Matplotlib |
| 8 | Violin Plot | Seaborn |
| 9 | Heatmap (Correlation Matrix) | Seaborn |
| 10 | Area Chart | Matplotlib |
| 11 | Subplots Dashboard | Matplotlib |

---
## 🔧 Step 1 — Import Libraries & Global Style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Global plot style ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'        : 120,
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
})

# Custom color palette
PALETTE = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']

print('✔ Libraries imported successfully')
print(f'  NumPy     : {np.__version__}')
print(f'  Pandas    : {pd.__version__}')
print(f'  Matplotlib: {plt.matplotlib.__version__}')
print(f'  Seaborn   : {sns.__version__}')

---
## 🗂️ Step 2 — Generate COVID-19 Dummy Dataset

In [ ]:
np.random.seed(42)

n_days    = 180
dates     = pd.date_range(start='2020-03-01', periods=n_days, freq='D')
countries = ['Pakistan', 'India', 'USA', 'UK', 'Germany']

# Simulate realistic wave-shaped case curves per country
def wave(n, peak, shift, noise=200):
    x = np.linspace(0, 2 * np.pi, n)
    return (np.sin(x + shift) * peak + peak + np.random.normal(0, noise, n)).clip(0)

# Build main DataFrame
data = pd.DataFrame({'Date': dates})
params = [(8000, 0.0), (25000, 0.3), (70000, 0.5), (15000, 0.2), (12000, 0.4)]

for country, (peak, shift) in zip(countries, params):
    data[country] = wave(n_days, peak, shift).astype(int)

# Additional columns
data['Deaths']       = (data['Pakistan'] * 0.018).astype(int)
data['Recoveries']   = (data['Pakistan'] * 0.72).astype(int)
data['Tests']        = (data['Pakistan'] * 1.5 + np.random.normal(0, 300, n_days)).clip(0).astype(int)
data['Positivity_%'] = (data['Pakistan'] / data['Tests'].replace(0, 1) * 100).round(2)

print(f'✔ Dataset ready: {data.shape[0]} rows x {data.shape[1]} columns')
print(f'  Period   : {dates[0].date()}  to  {dates[-1].date()}')
print(f'  Countries: {countries}')
data.head()

---
## 📈 Chart 1 — Line Plot (Time Series)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

ax.plot(data['Date'], data['Pakistan'], color=PALETTE[0], linewidth=2.5, label='Daily Cases')
ax.fill_between(data['Date'], data['Pakistan'], alpha=0.15, color=PALETTE[0])

ax.set_title('Pakistan — COVID-19 Daily Cases (Mar–Aug 2020)')
ax.set_xlabel('Date')
ax.set_ylabel('New Cases')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(fontsize=10)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
## 📈 Chart 2 — Multi-Line Plot (Country Comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for i, country in enumerate(countries):
    smoothed = data[country].rolling(7).mean()  # 7-day rolling average
    ax.plot(data['Date'], smoothed, color=PALETTE[i], linewidth=2, label=country)

ax.set_title('COVID-19 Daily Cases — Country Comparison (7-Day Rolling Average)')
ax.set_xlabel('Date')
ax.set_ylabel('New Cases (7-day avg)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(fontsize=10, loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
## 📊 Chart 3 — Bar Chart

In [ ]:
totals = {c: data[c].sum() for c in countries}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(totals.keys(), totals.values(), color=PALETTE[:5], edgecolor='white', linewidth=0.8)

# Value labels on top of bars
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 20000,
            f'{int(bar.get_height()):,}',
            ha='center', va='bottom', fontsize=9)

ax.set_title('Total COVID-19 Cases by Country (Mar–Aug 2020)')
ax.set_xlabel('Country')
ax.set_ylabel('Total Cases')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

---
## 📊 Chart 4 — Horizontal Bar Chart

In [ ]:
sorted_totals = dict(sorted(totals.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(list(sorted_totals.keys()), list(sorted_totals.values()),
        color=PALETTE[2], alpha=0.85, edgecolor='white')

for i, (k, v) in enumerate(sorted_totals.items()):
    ax.text(v + 5000, i, f'{v:,}', va='center', fontsize=9)

ax.set_title('Total Cases — Country Ranking (Low to High)')
ax.set_xlabel('Total Cases')
ax.set_ylabel('Country')
plt.tight_layout()
plt.show()

---
## 📊 Chart 5 — Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(data['Pakistan'], bins=30, color=PALETTE[0], edgecolor='white', alpha=0.85)

ax.axvline(data['Pakistan'].mean(),   color='red',    linestyle='--', lw=1.8,
           label=f"Mean: {data['Pakistan'].mean():.0f}")
ax.axvline(data['Pakistan'].median(), color='orange', linestyle='-.', lw=1.8,
           label=f"Median: {data['Pakistan'].median():.0f}")

ax.set_title('Distribution of Daily COVID-19 Cases — Pakistan')
ax.set_xlabel('Daily Cases')
ax.set_ylabel('Frequency (Number of Days)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 🔵 Chart 6 — Scatter Plot (Correlation)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(data['Tests'], data['Pakistan'],
                c=data['Deaths'], cmap='Reds',
                alpha=0.6, s=45, edgecolors='none')

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Deaths (color scale)', fontsize=9)

corr = data['Tests'].corr(data['Pakistan'])
ax.text(0.05, 0.95, f'Pearson r = {corr:.2f}',
        transform=ax.transAxes, fontsize=10, va='top', color='navy',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.set_title('Tests Conducted vs Daily Positive Cases')
ax.set_xlabel('Daily Tests Conducted')
ax.set_ylabel('Daily Positive Cases')
plt.tight_layout()
plt.show()

---
## 📦 Chart 7 — Box Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

box_data = [data[c].values for c in countries]
bp = ax.boxplot(box_data, labels=countries, patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 2},
                whiskerprops={'linewidth': 1.5},
                capprops={'linewidth': 1.5})

for patch, color in zip(bp['boxes'], PALETTE[:5]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_title('Daily Cases Distribution by Country (Box Plot)')
ax.set_xlabel('Country')
ax.set_ylabel('Daily Cases')
plt.tight_layout()
plt.show()

---
## 🎻 Chart 8 — Violin Plot

In [ ]:
melted = data[countries].melt(var_name='Country', value_name='Daily Cases')

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=melted, x='Country', y='Daily Cases',
               palette=PALETTE[:5], inner='quartile', ax=ax)

ax.set_title('Daily Cases Density Distribution — Violin Plot')
ax.set_xlabel('Country')
ax.set_ylabel('Daily Cases')
plt.tight_layout()
plt.show()

---
## 🌡️ Chart 9 — Heatmap (Correlation Matrix)

In [ ]:
corr_cols   = countries + ['Deaths', 'Recoveries', 'Tests', 'Positivity_%']
corr_matrix = data[corr_cols].corr()
mask        = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask,
            annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})

ax.set_title('Correlation Matrix — COVID-19 Variables', pad=15)
plt.tight_layout()
plt.show()

---
## 📉 Chart 10 — Area Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(data['Date'], data['Pakistan'],   alpha=0.5,  color=PALETTE[1], label='Cases')
ax.fill_between(data['Date'], data['Recoveries'], alpha=0.5,  color=PALETTE[2], label='Recoveries')
ax.fill_between(data['Date'], data['Deaths'],     alpha=0.85, color='#37474F',  label='Deaths')

ax.set_title('Pakistan — COVID-19 Cases, Recoveries & Deaths (Area Chart)')
ax.set_xlabel('Date')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(fontsize=10, loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
## 🖼️ Chart 11 — Subplots Dashboard (4 Charts in One Panel)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('COVID-19 Pakistan — Summary Dashboard',
             fontsize=15, fontweight='bold', y=1.01)

# Panel 1: Line
axes[0, 0].plot(data['Date'], data['Pakistan'].rolling(7).mean(), color=PALETTE[0], lw=2)
axes[0, 0].set_title('Daily Cases (7-Day Avg)')
axes[0, 0].set_ylabel('Cases')
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b'))

# Panel 2: Monthly Bar
monthly = data.set_index('Date')['Pakistan'].resample('M').sum()
axes[0, 1].bar(monthly.index.strftime('%b'), monthly.values, color=PALETTE[2], alpha=0.85, edgecolor='white')
axes[0, 1].set_title('Monthly Total Cases')
axes[0, 1].set_ylabel('Cases')

# Panel 3: Scatter
axes[1, 0].scatter(data['Tests'], data['Positivity_%'], alpha=0.4, color=PALETTE[3], s=20)
axes[1, 0].set_title('Tests vs Positivity Rate')
axes[1, 0].set_xlabel('Daily Tests')
axes[1, 0].set_ylabel('Positivity %')

# Panel 4: Histogram
axes[1, 1].hist(data['Pakistan'], bins=25, color=PALETTE[4], edgecolor='white', alpha=0.85)
axes[1, 1].axvline(data['Pakistan'].median(), color='red', linestyle='--', lw=1.5, label='Median')
axes[1, 1].set_title('Case Distribution')
axes[1, 1].set_xlabel('Daily Cases')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

---
## ✅ Visualization Summary

| # | Chart | Library | Insight |
|---|---|---|---|
| 1 | Line Plot | Matplotlib | Daily cases trend over time |
| 2 | Multi-Line Plot | Matplotlib | Country-wise comparison |
| 3 | Bar Chart | Matplotlib | Total cases per country |
| 4 | Horizontal Bar | Matplotlib | Country ranking |
| 5 | Histogram | Matplotlib | Case frequency distribution |
| 6 | Scatter Plot | Matplotlib | Tests vs cases correlation |
| 7 | Box Plot | Matplotlib | Spread and outliers per country |
| 8 | Violin Plot | Seaborn | Full density distribution |
| 9 | Heatmap | Seaborn | Variable correlation matrix |
| 10 | Area Chart | Matplotlib | Cases, recoveries, deaths overlap |
| 11 | Subplots Dashboard | Matplotlib | 4-chart summary panel |